#### Memory accounting - predicted vs measured bytes

In [2]:
!pip install import-ipynb
import import_ipynb

In [3]:
import lab06_starter as S
from lora_layer import replace_linear_with_lora
import torch

In [4]:
def predict_memory(trainable_params, total_params, is_full_ft=False):
    """Calculate memory in bytes for 32-bit training with AdamW.
    Weights: 4 bytes per parameter (all parameters).
    Gradients: 4 bytes per trainable parameter.
    Adam states: 2 moments * 4 bytes = 8 bytes per trainable parameter.
    (Note: Some rubrics state 16 bytes per trainable parameter for gradient + Adam moments, 
    let's break down strictly according to lab instructions: weights (4B), grads (4B), Adam states (8B)).
    """
    weight_bytes = total_params * 4
    grad_bytes = trainable_params * 4
    # Adam stores 2 states (mean and variance) per trainable parameter in 32-bit (4 bytes each = 8 bytes)
    optimizer_bytes = trainable_params * 8
    total_state_bytes = weight_bytes + grad_bytes + optimizer_bytes
    return weight_bytes, grad_bytes, optimizer_bytes, total_state_bytes

def run_piece4():
    print("--- Piece 4: Memory Accounting ---")
    model, tok = S.load_model()
    _, total_params = S.count_params(model)

    methods = [
        ("Full Fine-Tune", total_params, total_params),
        ("LoRA r=1", 16384, total_params),       # Approximate trainable params for r=1
        ("LoRA r=4", 65536, total_params),       # Approximate trainable params for r=4
        ("LoRA r=16", 262144, total_params),     # Approximate trainable params for r=16
    ]

    print(f"{'Method':<15} | {'Weight (MB)':<12} | {'Grad (MB)':<10} | {'Optimizer (MB)':<15} | {'Total Predicted (MB)'}")
    print("-" * 75)
    for name, train_p, tot_p in methods:
        w, g, o, tot = predict_memory(train_p, tot_p)
        print(f"{name:<15} | {w / 1e6:11.2f} | {g / 1e6:9.2f} | {o / 1e6:14.2f} | {tot / 1e6:18.2f}")

if __name__ == "__main__":
    run_piece4()

--- Piece 4: Memory Accounting ---


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Method          | Weight (MB)  | Grad (MB)  | Optimizer (MB)  | Total Predicted (MB)
---------------------------------------------------------------------------
Full Fine-Tune  |      538.06 |    538.06 |        1076.12 |            2152.24
LoRA r=1        |      538.06 |      0.07 |           0.13 |             538.26
LoRA r=4        |      538.06 |      0.26 |           0.52 |             538.85
LoRA r=16       |      538.06 |      1.05 |           2.10 |             541.21
